In [1]:

import numpy as np
import pandas as pd

from coin_flip_with_riskless_asset_model import CoinFlipWithRisklessAssetModel
from plotting_data import generate_arithmetic_vs_geometric_data
from plotting_functions import create_wealth_plot, create_empirical_growth_rates_plot, create_arithmetic_vs_geometric_plot

from finlib.ensemble_of_returns_paths import EnsembleOfReturnsPaths
from finlib.simulating_performance import crp_performance



In [2]:
import sys
import finlib   # does this work right now, in this same kernel?
import blogkit.brand_plotly as bp
print(sys.executable)

ModuleNotFoundError: No module named 'blogkit'

In [ ]:
coin_flip_model = CoinFlipWithRisklessAssetModel(gamma_heads=2.0, alpha=0.80, r=0.97, p=0.5)

In [ ]:
weights_for_risky_asset = np.array([0.0, 1.0])


In [ ]:
opt_result_for_CRP = coin_flip_model.solve_growth_rate_maximization_problem()

f_star = opt_result_for_CRP.x
weights_for_optimal_CRP = np.array([1 - f_star, f_star])

In [ ]:
##### CREATE PERFORMANCE ENSEMBLES FOR THE RISKY ASSET AND THE OPTIMAL CRP STRATEGY #####

seed = 12345
rng = np.random.default_rng(seed)

num_paths = 200
num_periods = 5000

gross_returns_tensor = coin_flip_model.generate_random_gross_returns(rng, num_periods=num_periods, num_paths=num_paths)

perf_risky_matrix = crp_performance(weights_for_risky_asset, gross_returns_tensor)
perf_optimal_CRP_matrix = crp_performance(weights_for_optimal_CRP, gross_returns_tensor)

ensemble_risky = EnsembleOfReturnsPaths.from_gross(perf_risky_matrix)
ensemble_optimal_CRP = EnsembleOfReturnsPaths.from_gross(perf_optimal_CRP_matrix)


In [ ]:
##### PLOTTING PREPRARATION #####

growth_rate_summary_df_risky = ensemble_risky.summarize_across_paths(ensemble_risky.running_growth_rate, threshold=0.0) # threshold is zero because we want to know the fraction of paths with positive growth rate at each period
wealth_summary_df_risky = ensemble_risky.summarize_across_paths(ensemble_risky.running_wealth_ratio, threshold=1.0) # threshold is 1 because we want to know the fraction of paths with a positive net return.
arith_geo_df = generate_arithmetic_vs_geometric_data(coin_flip_model)


In [ ]:
arith_geo_df

In [ ]:
create_arithmetic_vs_geometric_plot(arith_geo_df, opt_result_for_CRP)